In [4]:
import sys
import os

# Add scripts directory to path
scripts_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts'))
sys.path.insert(0, scripts_dir)

# Import functions directly from modules
import importlib.util

# Load qa_extract module
spec = importlib.util.spec_from_file_location("qa_extract", os.path.join(scripts_dir, "qa_extract.py"))
qa_extract = importlib.util.module_from_spec(spec)
spec.loader.exec_module(qa_extract)

import pandas as pd
import time

# Import functions from script
extract_video_metadata = qa_extract.extract_video_metadata
extract_qa_pairs_bilingual = qa_extract.extract_qa_pairs_bilingual
save_qa_to_csv = qa_extract.save_qa_to_csv
extract_qa_parallel = qa_extract.extract_qa_parallel

# Prototype 3: Q&A Extraction from Video Chapters & Captions

This notebook extracts question-answer pairs from videos using:
- Chapter titles as questions
- Captions (Hindi & English) as answers
- Timestamps for each Q&A pair

It reads video links from the output of Prototype 2 and processes each video to extract structured Q&A data.

In [5]:
# Import functions from script
extract_video_metadata = qa_extract.extract_video_metadata
extract_qa_pairs_bilingual = qa_extract.extract_qa_pairs_bilingual
save_qa_to_csv = qa_extract.save_qa_to_csv

print("✅ All functions imported from scripts/qa_extract.py")

✅ All functions imported from scripts/qa_extract.py


In [6]:
# Read video links from Prototype 2 output
video_links_df = pd.read_csv("../data/video_links.csv")

print(f"📊 Loaded {len(video_links_df)} video links")
print(f"Videos to process: {len(video_links_df['video_url'].unique())}\n")

# Get unique videos (some videos may appear in multiple playlists)
unique_videos = video_links_df[['video_url', 'video_id']].drop_duplicates().reset_index(drop=True)

print(f"🎯 Processing {len(unique_videos)} unique videos...")

📊 Loaded 1112 video links
Videos to process: 819

🎯 Processing 819 unique videos...


In [ ]:
# 🔍 VALIDATION: Test extraction with 3 sample videos
print(f"{'='*70}")
print(f"🔍 VALIDATION TEST: Extracting Q&A from 3 sample videos")
print(f"{'='*70}\n")

# Get 3 sample videos
test_videos = unique_videos.head(3).copy()

for test_idx, (_, video_row) in enumerate(test_videos.iterrows(), start=1):
    video_url = video_row['video_url']
    video_id = video_row['video_id']
    
    print(f"\n{'─'*70}")
    print(f"📺 TEST {test_idx}/3: {video_id}")
    print(f"URL: {video_url}")
    print(f"{'─'*70}\n")
    
    try:
        # Step 1: Extract metadata
        print("Step 1️⃣  Extracting metadata (chapters & captions)...")
        metadata = extract_video_metadata(video_url)
        
        if not metadata:
            print("  ❌ FAILED: Could not extract metadata")
            continue
        
        print(f"  ✅ Metadata extracted")
        
        # Step 2: Check chapters
        chapters = metadata['chapters']
        print(f"\nStep 2️⃣  Chapters found: {len(chapters)}")
        if chapters:
            for i, ch in enumerate(chapters[:3], 1):
                print(f"    Chapter {i}: {ch.get('title', 'N/A')} ({ch.get('start_time', 'N/A')}s)")
            if len(chapters) > 3:
                print(f"    ... and {len(chapters) - 3} more chapters")
        else:
            print("  ⚠️  WARNING: No chapters found!")
        
        # Step 3: Check Hindi captions
        captions_hi = metadata['captions_hi']
        print(f"\nStep 3️⃣  Hindi captions: {len(captions_hi)} text entries")
        if captions_hi:
            print(f"    First caption sample:")
            first_cap = captions_hi[0]
            if hasattr(first_cap, 'attrs'):
                print(f"      Start: {first_cap.get('start', 'N/A')}")
                print(f"      Text (first 100 chars): {first_cap.text[:100]}")
        else:
            print("  ⚠️  WARNING: No Hindi captions found!")
        
        # Step 4: Check English captions
        captions_en = metadata['captions_en']
        print(f"\nStep 4️⃣  English captions: {len(captions_en)} text entries")
        if captions_en:
            print(f"    First caption sample:")
            first_cap = captions_en[0]
            if hasattr(first_cap, 'attrs'):
                print(f"      Start: {first_cap.get('start', 'N/A')}")
                print(f"      Text (first 100 chars): {first_cap.text[:100]}")
        else:
            print("  ⚠️  WARNING: No English captions found!")
        
        # Step 5: Extract Q&A pairs
        if chapters and (captions_hi or captions_en):
            print(f"\nStep 5️⃣  Extracting Q&A pairs...")
            qa_pairs = extract_qa_pairs_bilingual(
                video_url, 
                metadata['title'], 
                chapters, 
                captions_hi, 
                captions_en
            )
            print(f"  ✅ Extracted {len(qa_pairs)} Q&A pairs")
            
            if qa_pairs:
                sample_qa = qa_pairs[0]
                print(f"\n  📋 Sample Q&A (First):")
                print(f"    Q: {sample_qa['question'][:80]}")
                print(f"    A (Hindi): {sample_qa['answer_hindi'][:100]}...")
                print(f"    A (English): {sample_qa['answer_english'][:100]}...")
        else:
            print(f"\nStep 5️⃣  ⚠️  SKIPPED: Missing chapters or captions")
        
        print(f"\n✅ TEST {test_idx} PASSED")
        
    except Exception as e:
        print(f"  ❌ ERROR in test {test_idx}: {type(e).__name__}: {str(e)[:100]}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*70}")
print(f"✅ VALIDATION TEST COMPLETE")
print(f"{'='*70}")
print(f"\n⚠️  Review the results above:")
print(f"  - Do chapters appear? ✓")
print(f"  - Do Hindi captions appear? ✓")
print(f"  - Do English captions appear? ✓")
print(f"  - Are Q&A pairs extracted? ✓")
print(f"\nIf all checks pass, proceed to BATCH PROCESSING cell.")
print(f"If any fail, check error messages and adjust extraction logic.\n")

In [7]:
# ⚡ BATCHED PARALLEL VIDEO PROCESSING WITH INCREMENTAL CSV SAVING
# Process videos in batches to avoid data loss

import os
from pathlib import Path

# Configuration
max_workers = 5           # Parallel workers
batch_size = 60           # Videos per batch  
sleep_between_batches = 5 # Seconds
output_csv = "../data/qa_dataset.csv"

# Ensure output directory exists
Path(output_csv).parent.mkdir(parents=True, exist_ok=True)

# Initialize CSV with headers if it doesn't exist
csv_initialized = os.path.exists(output_csv)

print(f"{'='*70}")
print(f"⚡ BATCHED PARALLEL PROCESSING WITH INCREMENTAL SAVING")
print(f"{'='*70}")
print(f"📊 Total videos: {len(unique_videos)}")
print(f"📦 Batch size: {batch_size}")
print(f"🔄 Parallel workers: {max_workers}")
print(f"⏸️  Sleep between batches: {sleep_between_batches}s")
print(f"💾 Output file: {output_csv}")
print(f"{'='*70}\n")

start_time = time.time()
all_failed_videos = []
total_qa_pairs = 0
batch_number = 0

# Process videos in batches
for batch_start in range(0, len(unique_videos), batch_size):
    batch_number += 1
    batch_end = min(batch_start + batch_size, len(unique_videos))
    batch_videos = unique_videos.iloc[batch_start:batch_end].copy()
    
    print(f"\n🔄 BATCH {batch_number}: Videos {batch_start + 1} to {batch_end} ({len(batch_videos)} videos)")
    print(f"{'─'*70}")
    
    batch_start_time = time.time()
    
    # Process this batch in parallel
    batch_qa_pairs, batch_failed = extract_qa_parallel(
        batch_videos,
        max_workers=max_workers,
        show_progress=True
    )
    
    # Append results to CSV incrementally
    if batch_qa_pairs:
        batch_df = pd.DataFrame(batch_qa_pairs)
        
        if csv_initialized:
            # Append to existing file
            batch_df.to_csv(output_csv, mode='a', header=False, index=False)
            print(f"\n✅ Appended {len(batch_df)} Q&A pairs to CSV")
        else:
            # Create new file with headers
            batch_df.to_csv(output_csv, mode='w', header=True, index=False)
            csv_initialized = True
            print(f"\n✅ Created CSV with {len(batch_df)} Q&A pairs")
        
        total_qa_pairs += len(batch_df)
    
    # Track failed videos
    all_failed_videos.extend(batch_failed)
    
    batch_elapsed = time.time() - batch_start_time
    print(f"⏱️  Batch time: {batch_elapsed:.1f}s | Total Q&A pairs: {total_qa_pairs}")
    
    # Sleep between batches (except after last batch)
    if batch_end < len(unique_videos):
        print(f"\n⏸️  Waiting {sleep_between_batches}s before next batch...")
        time.sleep(sleep_between_batches)

elapsed_time = time.time() - start_time

print(f"\n{'='*70}")
print(f"✅ BATCH PROCESSING COMPLETE!")
print(f"{'='*70}")
print(f"⏱️  Total processing time: {elapsed_time:.1f} seconds")
print(f"📊 Total videos processed: {len(unique_videos)}")
print(f"✅ Successful videos: {len(unique_videos) - len(all_failed_videos)}")
print(f"⚠️  Failed videos: {len(all_failed_videos)}")
print(f"📈 Total Q&A pairs extracted: {total_qa_pairs}")
print(f"💾 File saved: {output_csv}")
print(f"{'='*70}")

⚡ BATCHED PARALLEL PROCESSING WITH INCREMENTAL SAVING
📊 Total videos: 819
📦 Batch size: 60
🔄 Parallel workers: 5
⏸️  Sleep between batches: 5s
💾 Output file: ../data/qa_dataset.csv


🔄 BATCH 1: Videos 1 to 60 (60 videos)
──────────────────────────────────────────────────────────────────────

🚀 Starting parallel extraction with 5 workers
📊 Total videos to process: 60

[1/60] 📺 Processing: -aDTTy-o7vE


[2/60] 📺 Processing: -lS_YGR9ESo
[3/60] 📺 Processing: gHV3rttWaTE
[4/60] 📺 Processing: _ynawxVezT4[5/60] 📺 Processing: u4atgnKjT9I

[youtube] Extracting URL: https://www.youtube.com/watch?v=gHV3rttWaTE
[youtube] Extracting URL: https://www.youtube.com/watch?v=u4atgnKjT9I
[youtube] Extracting URL: https://www.youtube.com/watch?v=-aDTTy-o7vE
[youtube] Extracting URL: https://www.youtube.com/watch?v=_ynawxVezT4
[youtube] Extracting URL: https://www.youtube.com/watch?v=-lS_YGR9ESo
[youtube] u4atgnKjT9I: Downloading webpage
[youtube] -aDTTy-o7vE: Downloading webpage
[youtube] _ynawxVezT4: Downloading webpage
[youtube] gHV3rttWaTE: Downloading webpage
[youtube] -lS_YGR9ESo: Downloading webpage
[youtube] -aDTTy-o7vE: Downloading tv client config
[youtube] -lS_YGR9ESo: Downloading tv client config
[youtube] u4atgnKjT9I: Downloading tv client config
[youtube] gHV3rttWaTE: Downloading tv client config
[youtube] _ynawxVezT4: Downloading tv client config
[youtube] -aDTTy-o7vE: Downloading player 8

         player = https://www.youtube.com/s/player/8a6e7bc4/player_es6.vflset/en_US/base.js


[youtube] _ynawxVezT4: Downloading ios player API JSON


         n = fiJ_W02DScsNFxQ4 ; player = https://www.youtube.com/s/player/8a6e7bc4/player_es6.vflset/en_US/base.js
         player = https://www.youtube.com/s/player/8a6e7bc4/player_es6.vflset/en_US/base.js
         player = https://www.youtube.com/s/player/8a6e7bc4/player_es6.vflset/en_US/base.js


...

         n = CCjfp40V2zyEMBhc ; player = https://www.youtube.com/s/player/8a6e7bc4/player_es6.vflset/en_US/base.js


[youtube] VVfHMxuMDc4: Downloading m3u8 information
[youtube] Vtek4LEFpsY: Downloading player 8a6e7bc4
[youtube] XPmNxF938oY: Downloading player 8a6e7bc4
[youtube] Vtek4LEFpsY: Downloading tv player API JSON
[youtube] XPmNxF938oY: Downloading tv player API JSON
[youtube] Vtek4LEFpsY: Downloading ios player API JSON
[youtube] XPmNxF938oY: Downloading ios player API JSON


         player = https://www.youtube.com/s/player/8a6e7bc4/player_es6.vflset/en_US/base.js
         n = U8Olnx3FV01OnB1y ; player = https://www.youtube.com/s/player/8a6e7bc4/player_es6.vflset/en_US/base.js
         player = https://www.youtube.com/s/player/8a6e7bc4/player_es6.vflset/en_US/base.js
         n = gIFHWh5eF-V1qB7g ; player = https://www.youtube.com/s/player/8a6e7bc4/player_es6.vflset/en_US/base.js


[youtube] Vtek4LEFpsY: Downloading m3u8 information
[youtube] XPmNxF938oY: Downloading m3u8 information


KeyboardInterrupt: 

In [ ]:
# Display failed videos summary
if all_failed_videos:
    print(f"\n⚠️  FAILED VIDEOS REPORT:")
    print(f"{'─'*70}")
    for video_id, reason in all_failed_videos[:15]:
        print(f"  ❌ {video_id}: {reason}")
    if len(all_failed_videos) > 15:
        print(f"  ... and {len(all_failed_videos) - 15} more")
    print(f"{'─'*70}\n")
else:
    print(f"\n🎉 All videos processed successfully!")

# Load and display the saved CSV
qa_df = pd.read_csv(output_csv)
print(f"\n📋 Dataset Summary:")
print(f"Total rows in CSV: {len(qa_df)}")
print(f"Columns: {list(qa_df.columns)}")
print(f"\nFirst few rows:")
qa_df.head(10)

NameError: name 'all_qa_pairs' is not defined

In [ ]:
# Display sample results
print("\n📋 Sample Q&A pairs:")
print(f"\nTotal rows: {len(qa_df)}")
qa_df.head(10)